# SerpentGuard AI — Combine Sources & Re-clean

Merges the species-labelled photos from `08_extended_sources` with our processed iNaturalist dataset. Every new photo is checked against all 181,231 of our photos, in every split, and against the other new photos:
1. same iNaturalist observation or photo id as one of ours: dropped
2. perceptual hash within 8 bits of one of ours (the 64-bit DCT hash that `03_process_dataset` stored as `phash`): dropped
3. near-duplicates among the new photos: the largest file is kept

Survivors go to **train only**, so validation and test stay exactly as published. A control shrinks and re-encodes 20 of our own test photos and checks that step 2 catches every one. Saves `<your username>/serpentguard-extended-combined` with `combined_splits.csv`, `dropped.csv` and `combine_report.json`.

In [1]:
import io, json, os, logging
import numpy as np, pandas as pd
from pathlib import Path
from PIL import Image
from scipy.fft import dctn
os.environ["TQDM_DISABLE"] = "1"
import kagglehub
logging.getLogger("kagglehub").setLevel(logging.ERROR)

PROCESSED_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes-processed"
EXTENDED_HANDLE = "puneetbajajiitm/serpentguard-extended-sources"
KAGGLE_USERNAME = kagglehub.whoami()["username"]
OUT = Path("/kaggle/working/serpentguard-extended-combined")
OUT.mkdir(parents=True, exist_ok=True)
UPLOAD = True

processed_root = Path(kagglehub.dataset_download(PROCESSED_HANDLE))
df_base = pd.read_csv(processed_root / "metadata.csv", dtype={"obs_id": str, "photo_id": str, "observer_id": str}, low_memory=False)
classes_df = pd.read_csv(processed_root / "classes.csv")
print(f"Base dataset: {len(df_base):,} rows")

extended_root = Path(kagglehub.dataset_download(EXTENDED_HANDLE))
df_ext = pd.read_csv(extended_root / "extended_manifest.csv", dtype={"obs_id": str, "photo_id": str})
print(f"Extended manifest: {len(df_ext):,} rows")
print(df_ext["source"].value_counts().to_string())

Base dataset: 181,231 rows
Extended manifest: 2,038 rows
source
indian_snake     1334
snakeclef2023     704


In [2]:
SYNONYMS = {
    "Xenochrophis piscator": "Fowlea piscator",
    "Xenochrophis vittatus": "Fowlea vittatus",
    "Trimeresurus malabaricus": "Craspedocephalus malabaricus",
    "Trimeresurus gramineus": "Craspedocephalus gramineus",
    "Trimeresurus trigonocephalus": "Craspedocephalus trigonocephalus",
    "Calloselasma rhodostoma": "Calloselasma rhodostoma",
    "Naja tripudians": "Naja naja",
}

OUR_SPECIES = set(classes_df["species"].tolist())

if len(df_ext) > 0:
    df_ext["species"] = df_ext["species"].map(lambda s: SYNONYMS.get(s, s))
    df_ext = df_ext[df_ext["species"].isin(OUR_SPECIES)]
    print(f"After taxonomy normalisation + species filter: {len(df_ext):,} rows")
    print("Species distribution in extended:")
    print(df_ext["species"].value_counts().head(20).to_string())

After taxonomy normalisation + species filter: 2,038 rows
Species distribution in extended:
species
Ptyas mucosa                    447
Daboia russelii                 354
Echis carinatus                 246
Ophiophagus hannah              179
Bungarus caeruleus              106
Spalerosophis atriceps          102
Coelognathus helena              96
Trimeresurus popeiorum           90
Pareas carinatus                 50
Sinomicrurus macclellandi        47
Fowlea piscator                  44
Indotyphlops braminus            33
Oreocryptophis porphyraceus      16
Oligodon arnensis                11
Chrysopelea ornata                9
Uropeltis ellioti                 9
Rhabdophis helleri                9
Protobothrops mucrosquamatus      9
Oligodon cinereus                 9
Ahaetulla prasina                 8


In [3]:
PHASH_SIZE, PHASH_KEEP = 32, 8   # the same hash as 03_process_dataset
MIN_BYTES, MIN_SHORT_SIDE = 5000, 100
dropped = []

def phash(image):
    gray = np.asarray(image.convert("L").resize((PHASH_SIZE, PHASH_SIZE), Image.LANCZOS), dtype=np.float32)
    low = dctn(gray, norm="ortho")[:PHASH_KEEP, :PHASH_KEEP].flatten()
    bits = low > np.median(low[1:])
    return f"{int(''.join('1' if b else '0' for b in bits), 2):016x}"

def resolve(file):
    for candidate in (extended_root / file, extended_root / "images" / file):
        if candidate.exists():
            return candidate
    return None

def describe(file):
    path = resolve(file)
    if path is None:
        return None, 0, 0, 0, None
    with Image.open(path) as im:
        return str(path), path.stat().st_size, im.width, im.height, phash(im)

def drop(frame, mask, reason):
    dropped.append(frame[mask].assign(drop_reason=reason))
    print(f"  {reason}: dropped {int(mask.sum())}")
    return frame[~mask]

df_ext["abs_path"], df_ext["bytes"], df_ext["width"], df_ext["height"], df_ext["phash"] = zip(*map(describe, df_ext["file"]))
df_ext = drop(df_ext, df_ext["abs_path"].isna(), "file missing")
df_ext = drop(df_ext, df_ext["bytes"] < MIN_BYTES, f"under {MIN_BYTES // 1000} KB")
df_ext = drop(df_ext, df_ext[["width", "height"]].min(axis=1) < MIN_SHORT_SIDE, f"short side under {MIN_SHORT_SIDE} px")
print(f"Readable new photos: {len(df_ext):,}")

  file missing: dropped 0
  under 5 KB: dropped 0
  short side under 100 px: dropped 0
Readable new photos: 2,038


In [4]:
NEAR_DUP_BITS = 8
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

has_hash = df_base["phash"].notna()
ours_hash = np.array([int(h, 16) for h in df_base.loc[has_hash, "phash"]], dtype=np.uint64)
ours_rows = df_base.loc[has_hash, ["path", "split", "species"]].reset_index(drop=True)
print(f"Our photos with a stored pHash: {len(ours_hash):,} of {len(df_base):,}")

def nearest(hash_hex, bank):
    bits = POPCOUNT[(bank ^ np.uint64(int(hash_hex, 16))).view(np.uint8).reshape(-1, 8)].sum(axis=1)
    i = int(bits.argmin())
    return int(bits[i]), i

# Control: shrunk, re-encoded copies of our own test photos must be caught
control = df_base[df_base["split"] == "test"].sample(20, random_state=0)
caught = 0
for row in control.itertuples():
    with Image.open(processed_root / "data" / row.path) as im:
        small = im.convert("RGB").resize((int(im.width * 0.8), int(im.height * 0.8)))
    buffer = io.BytesIO()
    small.save(buffer, "JPEG", quality=70)
    bits, i = nearest(phash(Image.open(buffer)), ours_hash)
    caught += bits <= NEAR_DUP_BITS and ours_rows.at[i, "path"] == row.path
print(f"Control: {caught} of {len(control)} re-encoded copies of our test photos caught")
assert caught == len(control), "the near-duplicate check misses copies of our own photos"

# 1. Same iNaturalist observation or photo id
ids = df_ext["obs_id"].fillna("").ne("") & df_ext["obs_id"].isin(set(df_base["obs_id"]))
ids |= df_ext["photo_id"].fillna("").ne("") & df_ext["photo_id"].isin(set(df_base["photo_id"]))
df_ext = drop(df_ext, ids, "same iNaturalist observation or photo as ours")

# 2. Near-duplicate of one of our photos, in any split
near = [nearest(h, ours_hash) for h in df_ext["phash"]]
df_ext["nearest_bits"] = [bits for bits, _ in near]
df_ext["nearest_split"] = [ours_rows.at[i, "split"] for _, i in near]
df_ext["nearest_path"] = [ours_rows.at[i, "path"] for _, i in near]
near_ours = df_ext["nearest_bits"] <= NEAR_DUP_BITS
print("Near-duplicates of ours by the split they would have leaked from:", df_ext.loc[near_ours, "nearest_split"].value_counts().to_dict())
df_ext = drop(df_ext, near_ours, f"within {NEAR_DUP_BITS} bits of one of our photos")

# 3. Near-duplicates among the new photos: keep the largest file
kept, repeat = [], pd.Series(False, index=df_ext.index)
for i in df_ext.sort_values("bytes", ascending=False).index:
    h = df_ext.at[i, "phash"]
    if kept and nearest(h, np.array(kept, dtype=np.uint64))[0] <= NEAR_DUP_BITS:
        repeat[i] = True
    else:
        kept.append(int(h, 16))
df_ext = drop(df_ext, repeat, "near-duplicate of another new photo")
print(f"New photos kept: {len(df_ext):,}")

Our photos with a stored pHash: 181,231 of 181,231
Control: 20 of 20 re-encoded copies of our test photos caught
  same iNaturalist observation or photo as ours: dropped 0
Near-duplicates of ours by the split they would have leaked from: {'train': 48, 'test_extra': 5, 'val': 1}
  within 8 bits of one of our photos: dropped 54
  near-duplicate of another new photo: dropped 213
New photos kept: 1,771


In [5]:
# All new photos go to train; validation and test stay as published
new_rows = df_ext.assign(split="train", dataset="extended", path=df_ext["file"])
base_rows = df_base.assign(dataset="processed", source="inaturalist")
keep_cols = ["dataset", "source", "split", "path", "species", "obs_id", "photo_id", "license", "attribution", "url",
             "width", "height", "bytes", "phash"]
df_combined = pd.concat([base_rows, new_rows[keep_cols]], ignore_index=True)
df_combined = df_combined.drop(columns=["class_id"], errors="ignore").merge(classes_df[["species", "class_id"]], on="species", how="left")
df_combined["class_id"] = df_combined["class_id"].fillna(-1).astype(int)

# Leakage checks against validation and test
eval_splits = ["val", "test", "test_extra"]
eval_obs = set(df_base.loc[df_base["split"].isin(eval_splits), "obs_id"].dropna())
obs_overlap = int(new_rows["obs_id"].fillna("").isin(eval_obs).sum())
eval_bank = ours_hash[ours_rows["split"].isin(eval_splits).values]
closest = min((nearest(h, eval_bank)[0] for h in new_rows["phash"]), default=64)
print(f"Leakage check: {obs_overlap} new photos share an observation with val/test; closest new photo is {closest} bits from any val/test photo")
assert obs_overlap == 0 and closest > NEAR_DUP_BITS

df_combined.to_csv(OUT / "combined_splits.csv", index=False)
dropped_df = pd.concat(dropped, ignore_index=True) if dropped else pd.DataFrame()
dropped_df.drop(columns=["abs_path"], errors="ignore").to_csv(OUT / "dropped.csv", index=False)
report = {"base_photos": len(df_base), "new_candidates": int(len(new_rows) + len(dropped_df)), "new_kept": len(new_rows),
          "dropped_by_reason": dropped_df["drop_reason"].value_counts().to_dict() if len(dropped_df) else {},
          "near_duplicate_bits": NEAR_DUP_BITS, "control_caught": f"{caught}/{len(control)}",
          "closest_new_to_eval_bits": closest, "kept_by_source": new_rows["source"].value_counts().to_dict(),
          "kept_by_species": new_rows["species"].value_counts().to_dict()}
(OUT / "combine_report.json").write_text(json.dumps(report, indent=1))
print(f"Combined splits saved: {len(df_combined):,} rows")
print(df_combined["split"].value_counts().to_string())

Leakage check: 0 new photos share an observation with val/test; closest new photo is 10 bits from any val/test photo
Combined splits saved: 183,002 rows
split
train           131012
val              25639
test_extra       18390
test              7305
unknown_pool       656


## Summary

In [6]:
df_final = pd.read_csv(OUT / "combined_splits.csv", low_memory=False)
print("=== Combined Dataset Summary ===")
print(f"Total rows:        {len(df_final):,}")
print(f"Unique species:    {df_final['species'].nunique()}")
print("Split distribution:")
print(df_final["split"].value_counts().to_string())
print("Source distribution:")
print(df_final["source"].value_counts().to_string())
print("New training photos by species:")
print(df_final[df_final["dataset"] == "extended"].groupby("species").size().sort_values(ascending=False).to_string())

(OUT / "README.md").write_text("""# SerpentGuard extended combined splits (private, course use only)

Made by notebooks/09_combine_and_reclean.ipynb. combined_splits.csv is our processed metadata plus the
new photos from serpentguard-extended-sources that survived the duplicate checks; new photos are in
train only. The dataset column says where each file lives: "processed" (data/<path> in
serpentguard-indian-snakes-processed) or "extended" (<path> in serpentguard-extended-sources).
dropped.csv lists every removed candidate with its reason; combine_report.json has the totals.
""")
if UPLOAD:
    kagglehub.dataset_upload(f"{KAGGLE_USERNAME}/serpentguard-extended-combined", str(OUT), version_notes="09_combine_and_reclean")

=== Combined Dataset Summary ===
Total rows:        183,002
Unique species:    289
Split distribution:
split
train           131012
val              25639
test_extra       18390
test              7305
unknown_pool       656
Source distribution:
source
inaturalist      181231
indian_snake       1118
snakeclef2023       653
New training photos by species:
species
Daboia russelii                 332
Ptyas mucosa                    296
Echis carinatus                 218
Ophiophagus hannah              179
Bungarus caeruleus              102
Spalerosophis atriceps           94
Coelognathus helena              90
Trimeresurus popeiorum           88
Pareas carinatus                 48
Sinomicrurus macclellandi        47
Fowlea piscator                  44
Indotyphlops braminus            27
Oreocryptophis porphyraceus      16
Oligodon cinereus                 9
Uropeltis ellioti                 9
Rhabdophis helleri                9
Protobothrops mucrosquamatus      8
Xenopeltis unicolor     